# Chaining & Runnables

## Load ENV file

In [1]:
from dotenv import load_dotenv
import os

load = load_dotenv('./../.env', override=True)

print(os.getenv('LANGSMITH_PROJECT'))

EALangchainTrainingsTest


## Create cloud LLM object

In [4]:
from langchain_ollama import ChatOllama
import os

ollama_cloud_llm = ChatOllama(
    base_url="http://localhost:11434/",  # Ollama cloud endpoint
    model="gemini-3-flash-preview:cloud", #gemini-3-flash-preview:cloud #qwen3.5:cloud
    temperature=0.5,
    max_tokens=450,
    headers={
        "Authorization": f"Bearer {os.getenv('OLLAMA_CLOUD_API_KEY')}"  # Cloud auth
    }
)

ollama_local_llm_1 = ChatOllama(
    base_url="http://localhost:11434/",
    model="llama3.2:latest ",
    temperature=0.5,
    max_tokens=450
)

ollama_local_llm_2 = ChatOllama(
    base_url="http://localhost:11434/",
    model="qwen2.5:7b",
    temperature=0.5,
    max_tokens=450
)

## Create OpenAI LLM object

In [51]:
from langchain_openai import ChatOpenAI
import os

openai_llm = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0.5,
    #max_tokens=500,
)

## Understanding Chaing & Runnables

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate)

prompt_template = ChatPromptTemplate([
    ("system", "You are an LLM expert"),
    ("human", "What is the advantage of running the LLM in {env}?")
])

chain = prompt_template | ollama_cloud_llm

chain.invoke({"env": "local machine"})

AIMessage(content='Running an LLM on your local machine (instead of in the cloud) offers several clear advantages:\n\n- Privacy and data security\n  - Your prompts and outputs stay on-premises, reducing risk of data leakage or vendor access.\n  - Easier to enforce strict data-handling policies and retention rules.\n\n- Data sovereignty and compliance\n  - You can ensure data residency and meet industry/regulatory requirements (e.g., HIPAA, GDPR) with auditable controls.\n\n- Lower latency and offline operation\n  - For some use cases, you get faster responses since there’s no network round-trip to a remote API.\n  - Can run offline or in environments with limited or unreliable internet access.\n\n- Cost predictability and potential savings\n  - No per-token API fees or usage caps; you pay for hardware once (though there are ongoing energy and maintenance costs).\n  - Scales with your own infrastructure, which can be cheaper for heavy or steady usage.\n\n- Greater control and customizat

### String Parsing

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser

# 2nd way
prompt_template = ChatPromptTemplate([
    ("system", "You are an LLM expert"),
    ("human", "What is the advantage of running the LLM in {env}?")
])

#chaining mechanism
chain = prompt_template | ollama_cloud_llm | StrOutputParser()

response = chain.invoke({"env": "local machine"})

print(response)

As an LLM expert, I see the shift toward "Local LLMs" (often called Edge AI or On-Premise Inference) as one of the most significant trends in the field. While cloud APIs (like OpenAI or Anthropic) offer convenience and massive scale, running models locally on your own hardware (Mac M-series, NVIDIA GPUs, or even high-end CPUs) provides several strategic advantages.

Here is a breakdown of the primary advantages of running LLMs locally:

### 1. Data Privacy and Security
This is the single most important factor for enterprises and individuals handling sensitive data.
*   **Zero Data Leakage:** When you use a cloud API, your prompts and data are sent to a third-party server. Even with "enterprise" agreements, there is a residual risk of data breaches or accidental training on your inputs.
*   **Regulatory Compliance:** For industries like healthcare (HIPAA), finance (SEC), or legal, local execution ensures that sensitive information never leaves the local network, simplifying compliance a

### Chaining Multiple Chains (Sequential)

In [3]:
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import StrOutputParser

# 2nd way
prompt_template = ChatPromptTemplate([
    ("system", "You are an LLM expert"),
    ("human", "What is the advantage of running the LLM in {env}?")
])

#Chain 1
detailedResponseChain = prompt_template | ollama_cloud_llm | StrOutputParser()

headingInfoTemplate = ChatPromptTemplate.from_template("""
Analyse the response and just get the headings or sub-headings from {response}
                                         
Response should be in bullet points.

""")

#Chain 2
headingChain = {"response": detailedResponseChain} | headingInfoTemplate | ollama_cloud_llm | StrOutputParser()

short_response = headingChain.invoke({"env": "local machine"})

print(short_response)

ResponseError: this model requires a subscription, upgrade for access: https://ollama.com/upgrade (ref: 3c4352d1-4143-4f4d-b4c7-90fc0a963616) (status code: 403)

### Chaining Multiple Chains (Parallel)

#### Running dependent tasks parallely and then combining the results.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

prompt_template = ChatPromptTemplate([
    ("system", "You are an LLM expert"),
    ("human", "What is the advantage of running the LLM in {env}?")
])

#Chain 1
detailedResponseChain = prompt_template | ollama_cloud_llm | StrOutputParser()

headingInfoTemplate = ChatPromptTemplate.from_template("""
Analyse the response and just get the headings or sub-headings from {response}
                                         
Response should be in bullet points.

""")

#Chain 2
headingChain = {"response": detailedResponseChain} | headingInfoTemplate | openai_llm | StrOutputParser()

runnableParallel = RunnableParallel(chain1=detailedResponseChain, chain2=headingChain)

response = runnableParallel.invoke({"env": "local machine"})

print(response['chain1'])
print("\n------------------------------------------------------------------------------------------\n")
print(response['chain2'])

As an LLM expert, I see the shift toward "Local LLMs" (running models on your own hardware using tools like Ollama, LM Studio, or vLLM) as one of the most significant trends in the industry.

While cloud APIs (OpenAI, Anthropic) offer massive scale, running a model locally provides several strategic advantages:

### 1. Data Privacy and Security
This is the primary driver for most enterprises and individuals.
*   **Zero Data Leakage:** When you use a cloud API, your prompts and sensitive data are sent to a third-party server. Even with "privacy modes," there is a residual risk.
*   **Compliance:** For industries like healthcare (HIPAA), finance, or legal, local execution ensures that sensitive PII (Personally Identifiable Information) never leaves the local network, simplifying compliance audits.

### 2. Cost Efficiency (Zero Token Fees)
Cloud LLMs charge "per token" (roughly per word). While cheap for a single query, this adds up for high-volume tasks.
*   **Sunk Cost vs. Variable Cost

#### Running independent tasks parallely and then combining the results.

In [61]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

localMachineTemplate = ChatPromptTemplate([
    ("system", "You are an LLM expert"),
    ("human", "What is the advantage of running the LLM in {env1}?")
])

#Chain 1
localMachineResponseChain = localMachineTemplate | ollama_local_llm_1 | StrOutputParser()

cloudMachineTemplate = ChatPromptTemplate.from_template("""
What is the advantage of running the LLM in {env2}?
""")

#Chain 2
cloudMachineResponseChain = cloudMachineTemplate | ollama_local_llm_2 | StrOutputParser()

runnableParallel = RunnableParallel(chain1=localMachineResponseChain, chain2=cloudMachineResponseChain)

response = runnableParallel.invoke({"env1": "local machine", "env2": "cloud machine"})

print(response['chain1'])
print("\n------------------------------------------------------------------------------------------\n")
print(response['chain2'])

Running a Large Language Model (LLM) on a local machine, also known as "edge inference" or "on-device processing", has several advantages:

1. **Reduced latency**: By processing the model locally, you can avoid sending data to a remote server for inference, which reduces latency and improves responsiveness.
2. **Improved security**: Storing sensitive data, such as user input or model weights, on a local machine can reduce the risk of data breaches or unauthorized access.
3. **Increased privacy**: Local processing allows you to maintain control over user data and ensure that it is not transmitted to third-party servers for analysis or advertising purposes.
4. **Lower bandwidth usage**: By performing inference locally, you can significantly reduce the amount of data transferred between devices and servers, which can be particularly useful for applications with limited internet connectivity.
5. **Faster model updates**: With local processing, you can update your LLM model in real-time wit

### Runnable Lambda

In [59]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

prompt_template = ChatPromptTemplate([
    ("system", "You are an LLM expert"),
    ("human", "What is the advantage of running the LLM in {env}?")
])

#Chain 1
detailedResponseChain = prompt_template | ollama_local_llm_1 | StrOutputParser()

headingInfoTemplate = ChatPromptTemplate.from_template("""
Analyse the response and just get the headings or sub-headings from {response}
                                         
Response should be in bullet points.

""")

def choose_llm(response):
    response_text = str(response)
    if len(response_text) < 300:
        return ollama_local_llm_2
    else:
        return ollama_local_llm_1
    
llm_selector = RunnableLambda(choose_llm)

#Chain 2
headingChain = {"response": detailedResponseChain} | headingInfoTemplate | llm_selector | StrOutputParser()

short_response = headingChain.invoke({"env": "local machine"})

print(short_response)

Here are the headings/sub-headings from the response:

* Advantages of Running a Large Language Model (LLM) Locally:
	+ Faster Execution
	+ Reduced Latency
	+ Data Privacy
	+ Security
	+ Lower Latency for Real-time Applications
	+ Better Performance in Resource-Constrained Environments
	+ Improved Model Performance
	+ Reduced Dependence on Internet Connectivity
* Challenges and Limitations of Running an LLM Locally:
	+ Increased Hardware Requirements
	+ Data Storage and Management
	+ Model Updates and Maintenance


### Using @Chain decorator

In [60]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain

prompt_template = ChatPromptTemplate([
    ("system", "You are an LLM expert"),
    ("human", "What is the advantage of running the LLM in {env}?")
])

#Chain 1
detailedResponseChain = prompt_template | ollama_local_llm_1 | StrOutputParser()

headingInfoTemplate = ChatPromptTemplate.from_template("""
Analyse the response and just get the headings or sub-headings from {response}
                                         
Response should be in bullet points.

""")

@chain
def choose_llm(response):
    response_text = str(response)
    if len(response_text) < 300:
        return ollama_local_llm_2
    else:
        return ollama_local_llm_1
    

#Chain 2
headingChain = {"response": detailedResponseChain} | headingInfoTemplate | choose_llm | StrOutputParser()

short_response = headingChain.invoke({"env": "local machine"})

print(short_response)

Here are the headings and sub-headings from the response:

* **Advantages of Running a Large Language Model (LLM) on a Local Machine**
	+ Reduced latency
	+ Improved security
	+ Increased control and customization
	+ Lower latency for real-time applications
	+ Reduced bandwidth usage
	+ Better handling of sensitive data
	+ Faster inference times
	+ Reduced dependence on internet connectivity
* **Challenges of Running an LLM on a Local Machine**
	+ Computational resources
	+ Data storage and management
	+ Maintenance and updates
